# Lab 044 — Resuming Execution

## Lab Intro

In the previous labs we learned how to:

- Pause execution using interrupts
- Wait for human review
- Modify workflow state during review

Now we learn how to:

```text
Resume Execution
```

after a workflow has been interrupted.

This is a core Human-in-the-Loop capability in LangGraph.

---

## Enterprise Analogy

Imagine a purchase approval process:

```text
Purchase Request
        ↓
Manager Review
        ↓
(waiting)
        ↓
Approved
        ↓
Process Purchase
```

The workflow pauses during review.

Once the manager responds, the workflow continues from the exact point where it stopped.

---

## Key Idea

An interrupt creates a pause:

```text
Workflow Running
       ↓
Interrupt
       ↓
Waiting
```

A resume command continues execution:

```text
Waiting
      ↓
Command(resume=...)
      ↓
Continue Execution
```

---

## Visual Model

```text
START
   ↓
review_request
   ↓
INTERRUPT
   ↓
(waiting)
   ↓
Command(resume=...)
   ↓
finalize
   ↓
END
```

---

## Lab Code

In [ ]:
from pydantic import BaseModel

from langgraph.graph import StateGraph
from langgraph.graph import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

# -------------------------
# State
# -------------------------

class State(BaseModel):
    request: str
    approval: str = ""
    result: str = ""

# -------------------------
# Human Review
# -------------------------

def review_request(state: State):

    decision = interrupt(
        "Approve or reject?"
    )

    return {
        "approval": decision
    }

# -------------------------
# Final Step
# -------------------------

def finalize(state: State):

    return {
        "result":
        f"Request was {state.approval}"
    }

# -------------------------
# Build Graph
# -------------------------

builder = StateGraph(State)

builder.add_node(
    "review_request",
    review_request
)

builder.add_node(
    "finalize",
    finalize
)

builder.add_edge(
    START,
    "review_request"
)

builder.add_edge(
    "review_request",
    "finalize"
)

builder.add_edge(
    "finalize",
    END
)

# -------------------------
# Checkpointer Required
# -------------------------

graph = builder.compile(
    checkpointer=InMemorySaver()
)

# -------------------------
# Thread Configuration
# -------------------------

config = {
    "configurable": {
        "thread_id": "approval-1"
    }
}

# -------------------------
# Initial Execution
# -------------------------

graph.invoke(
    {
        "request": "Purchase Laptop"
    },
    config=config
)

print("Workflow paused")

# -------------------------
# Resume Execution
# -------------------------

result = graph.invoke(
    Command(
        resume="approved"
    ),
    config=config
)

print(result)